In [3]:
import matplotlib.pyplot as plt
from matplotlib import rcParams 
from typing import Dict, List, Union
import numpy as np
from scipy.stats import norm

a)  Smart up our plotter to be able to customize the hard-coded
layout setup inside the function. Add more custom features that you
would like to use

In [4]:
def my_plotter(x: List[float], y: Union[List[float], List[List[float]]], layout: Dict = {}, names: List[str] = None) -> None:
    y = [y] if all(isinstance(item, float) for item in y) else y

    #setting the size from the layout with a default value included
    figsize = layout.get('figsize', (8, 4))
    plt.figure(figsize=figsize)

    lines = []
    show_legend = True if names is not None else False

    #setting the linestyle and linewidth from the layout with default values included
    plot_kwargs = {
        'linestyle': layout.get('linestyle', 'solid'),
        'linewidth': layout.get('linewidth', 4)
    }
    
    if names is not None:
        show_legend = True
        if len(names) != len(y):
            raise ValueError("Length of names is not matching with number of plotted y lists.")

    for i, y_item in enumerate(y):
        if show_legend:
            plot_kwargs['label'] = names[i]

            #custom colors can be used if they are included in the layout
            if 'colors' in layout and len(layout['colors']) == len(y):
                plot_kwargs['color'] = layout['colors'][i]

        _line = plt.plot(x, y_item, **plot_kwargs)
        lines.append(_line)

    if show_legend:
        #setting the fontsize from the layout with a default value included
        plt.legend(fontsize=layout.get('legend_fontsize', 16))

    if 'title' in layout:
        #setting the fontsize of the title from the layout with a default value included
        plt.title(layout['title'], fontsize=layout.get('title_fontsize', 20))
        #setting the spacing for the title from the layout with a default value included
        rcParams['axes.titlepad'] = layout.get('title_pad', 30)

    if 'x_label' in layout:
        #setting the fontsize of the label of the x axis from the layout with a default value included
        plt.xlabel(layout['x_label'], fontsize=layout.get('label_fontsize', 16))
        #setting the spacing for the label of the x axis from the layout with a default value included
        rcParams['axes.labelpad'] = layout.get('x_label_pad', 20)


    #in addition to the label of the x axis, we can add and customize a label for the y axis
    if 'y_label' in layout:
        plt.ylabel(layout['y_label'], fontsize=layout.get('label_fontsize', 16))

    # we can add a grid in the background of our figure
    if layout.get('grid', False):
        plt.grid(True, linestyle='--', alpha=0.7)

    ax = plt.gca() 
    ax.axhline(linestyle='--', color='black', linewidth=1)

    # we can save our figure, if the file name is given in the layout
    if 'save_path' in layout:
        plt.savefig(layout['save_path'], bbox_inches='tight')

    plt.show()


b) Upgrade the BSM call option pricer to be able to return the greeks
as well, not just the price.

In [6]:
def black_scholes_eur_call(r: float, T: float, S0: float, sigma: float, K: Union[float, List[float], np.ndarray]) -> np.ndarray:
    assert sigma > 0

    K_vec = np.array([K]) if isinstance(K, float) else np.array(K)

    d1_vec = ( np.log( S0 / K ) + ( r + 0.5 * sigma**2 ) * T ) / ( sigma * T**0.5 )
    d2_vec = d1_vec - sigma * T**0.5

    N_d1_vec = norm.cdf(d1_vec)
    N_d2_vec = norm.cdf(d2_vec)

    # N'_S(d1)
    pdf_d1_vec = norm.pdf(d1_vec) 

    price = N_d1_vec * S0 - K_vec * np.exp((-1.0)*r*T) * N_d2_vec

    delta = N_d1_vec
    gamma = pdf_d1_vec / (S0 * sigma * T**0.5)
    vega = S0 * pdf_d1_vec * T**0.5
    theta = -(S0 * pdf_d1_vec * sigma) / (2 * T**0.5) - r * K_vec * np.exp((-1.0) * r * T) * N_d2_vec
    rho = K_vec * T * np.exp((-1.0) * r * T) * N_d2_vec

    return {
        'price': price,
        'delta': delta,
        'gamma': gamma,
        'vega': vega,
        'theta': theta,
        'rho': rho
    }

c) Implement the BSM pricer for European put option.

In [7]:
def black_scholes_eur_put(r: float, T: float, S0: float, sigma: float, K: Union[float, List[float], np.ndarray]) -> np.ndarray:
    assert sigma > 0

    K_vec = np.array([K]) if isinstance(K, (float, int)) else np.array(K)

    d1_vec = ( np.log( S0 / K_vec ) + ( r + 0.5 * sigma**2 ) * T ) / ( sigma * T**0.5 )
    d2_vec = d1_vec - sigma * T**0.5

    N_minus_d1_vec = norm.cdf(-d1_vec)
    N_minus_d2_vec = norm.cdf(-d2_vec)

    return K_vec * np.exp((-1.0)*r*T) * N_minus_d2_vec - N_minus_d1_vec * S0

d) With the call and put pricer, check if Put-Call parity holds in
practice.

In [10]:
#test parameters
r_test = 0.05       #risk-free interest rate 
T_test = 1.0        #time to maturity in years
S0_test = 100.0     #initial price 
sigma_test = 0.2    #volatility
K_test = 100.0      #strike price

call_result = black_scholes_eur_call(r=r_test, T=T_test, S0=S0_test, sigma=sigma_test, K=K_test)
call_price = call_result['price'] if isinstance(call_result, (float, np.floating)) else call_result['price'][0]

put_result = black_scholes_eur_put(r=r_test, T=T_test, S0=S0_test, sigma=sigma_test, K=K_test)
put_price = put_result if isinstance(put_result, float) else put_result[0]

#left-hand side of Put-Call parity
lhs = call_price - put_price
#right-hand side of Put-Call parity
rhs = S0_test - K_test * np.exp(-r_test * T_test)

#verifying if Put-Call parity holds within a small numerical tolerance
parity = np.isclose(lhs, rhs, atol=1e-5)
print(parity)

True
